# Language Models From Scratch

A **large language model** (LLM) is a [Transformer](/courses/deep-learning/09-attention-transformers.html) trained to predict the next token — and from that single objective, instruction following, reasoning, and alignment all emerge. This series builds an LLM from scratch and follows it all the way: a 29.9M-parameter **nanoGPT** model with SwiGLU, RMSNorm, and RoPE layers, trained on [TinyShakespeare](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt), instruction-tuned with LoRA, aligned with DPO and GRPO, distilled, quantized, and finally deployed as a reasoning model that answers questions about Hamlet.

## About This Series

This series covers the [full pipeline]{.mark} of building a language model — from the first attention head through pretraining, alignment, and deployment — deriving every technique from first principles and implementing it in working code. It is organized in five parts that follow the natural lifecycle of an LLM: architecture design, pretraining at scale, alignment with human preferences, inference optimization, and a capstone that chains every stage into a single end-to-end system.

**Audience.** Someone who has completed the [Deep Learning Foundations](/courses/deep-learning/index.html) series (or equivalent) and wants to go from understanding backpropagation to being able to pretrain, fine-tune, and align a language model from scratch. A mathematics background is assumed — derivations are shown in full. Code is the other half of the explanation.

**Stack.** The series uses PyTorch 2.0+ throughout, with `torch.autocast` and `GradScaler` for mixed-precision training, `torchrun` for DDP and FSDP distributed training, and a from-scratch BPE tokenizer (validated against tiktoken). Diagnostics are built on forward/backward hooks and Matplotlib. No high-level training frameworks — every optimizer step, gradient accumulation loop, and checkpoint is written explicitly so the machinery is visible.

**The main project** is a [29.9M-parameter nanoGPT]{.mark} — a decoder-only Transformer with SwiGLU, RMSNorm, and RoPE — trained on [TinyShakespeare](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt) and carried through six sequential stages: pretraining, SFT cold start with LoRA, reward modeling, GRPO reinforcement learning, INT8 quantization, and evaluation. The end result is a reasoning model that answers questions about Hamlet. The project is deliberately small enough to run on a single GPU but architecturally faithful to the patterns used at scale — the same code structure, the same training diagnostics, the same alignment pipeline.

## Part I. Architecture

| # | Title | Key Topics |
|---|---|---|
| 01 | [GPT Architecture from Scratch](/courses/llm/01-gpt-architecture.html) | Scaled dot-product attention, causal masking, multi-head attention, RoPE, RMSNorm, SwiGLU FFN, `1/√2L` weight init, GPT block assembly |
| 02 | [Byte Pair Encoding & Tokenization](/courses/llm/02-tokenization.html) | BPE merge loop, encode/decode, vocabulary size tradeoffs, special tokens, `BPETokenizer` with save/load, tiktoken comparison |

: {tbl-colwidths="[6,32,62]"}

## Part II. Pretraining

| # | Title | Key Topics |
|---|---|---|
| 03 | [Data Pipelines for Pretraining](/courses/llm/03-data-pipelines.html) | `IterableDataset`, pack-and-split with EOS boundaries, shuffle buffer reservoir sampling, worker file-sharding, pre-tokenized shards, MinHash LSH deduplication |
| 04 | [Optimizer, Schedule & Mixed Precision](/courses/llm/04-training-loop.html) | AdamW, LR range test, gradient accumulation (`zero_grad` bug), BF16 vs FP16, `GradScaler`, cosine warmup schedule, Chinchilla scaling laws |
| 05 | [Gradient & Activation Diagnostics](/courses/llm/05-scaling-diagnostics.html) | Forward/backward hooks, `GradientMonitor`, gradient-to-weight ratio `ρ_l`, dead neuron detection, `LiveDashboard`, catalog of eight training pathologies |
| 06 | [Pretraining nanoGPT](/courses/llm/06-pretraining.html) | `torch.autocast`, `TrainingConfig`, `chinchilla_optimal()`, `plan_training_run()`, instrumented pretraining loop with checkpointing |
| 07 | [Multi-GPU with DDP & FSDP](/courses/llm/07-distributed-training.html) | Ring-allreduce, DDP gradient hooks, `no_sync()`, rank-sharded data loading, FSDP all-gather/reduce-scatter, `FULL_SHARD` vs `SHARD_GRAD_OP`, `torchrun` |

: {tbl-colwidths="[6,32,62]"}

## Part III. Alignment

| # | Title | Key Topics |
|---|---|---|
| 08 | [Instruction Tuning with LoRA](/courses/llm/08-sft-lora.html) | Masked cross-entropy on assistant tokens, chat template, `InstructDataset`, LoRA math (`W = W₀ + BA`), `LoRALinear.from_linear()`, `inject_lora()`, `merge_lora()` |
| 09 | [DPO & Reward Modeling](/courses/llm/09-preference-optimization.html) | DPO closed-form derivation (Z(x) cancellation), implicit reward `β log(π_θ/π_ref)`, Bradley-Terry reward model, best-of-N sampling, reward calibration |
| 10 | [Reinforcement Learning with GRPO](/courses/llm/10-grpo.html) | REINFORCE policy gradient, group-relative advantage via z-score (no value network), PPO clipped surrogate, KL penalty, `compute_group_advantages()`, reward hacking detection |

: {tbl-colwidths="[6,32,62]"}

## Part IV. Inference

| # | Title | Key Topics |
|---|---|---|
| 11 | [Knowledge Distillation](/courses/llm/11-distillation.html) | Soft targets with temperature τ, `τ²` gradient rescaling, forward vs. reverse KL, sequence-level KD, `FeatureAlignmentProjection`, DeepSeek-R1 style post-training distillation |
| 12 | [Model Quantization](/courses/llm/12-quantization.html) | Affine/symmetric quantization, per-tensor vs per-channel, `QuantizedLinear.from_linear()`, `ActivationCalibrator`, INT4 block quantization (`group_size=128`), activation outliers, QAT with STE |
| 13 | [KV Cache & Speculative Decoding](/courses/llm/13-inference.html) | `KVCache` prefill + decode, online softmax tiling, `F.scaled_dot_product_attention`, draft model verification, `FastInferenceEngine` combining all three |
| 14 | [Inference-Time Scaling](/courses/llm/14-inference-time-scaling.html) | Best-of-N (`1-(1-p)^N`), beam search with length normalization, majority voting, `ProcessRewardModel`, MCTS (select/expand/simulate/backpropagate), compute-quality tradeoff |

: {tbl-colwidths="[6,32,62]"}

## Part V. Capstone

| # | Title | Key Topics |
|---|---|---|
| 15 | [Training a Reasoning Model](/courses/llm/15-reasoning-model.html) | `pico`/`nano` configs, six sequential stages (pretraining → SFT cold start → reward model → GRPO → INT8 deployment → evaluation), per-stage sanity checks, Shakespeare Q&A |

: {tbl-colwidths="[6,32,62]"}

## DeepSeek

A three-notebook companion series on frontier architecture choices from DeepSeek-V2/V3 and the Engram memory layer. DeepSeek-V2 introduced Multi-head Latent Attention (MLA) — projecting keys and values through a low-rank bottleneck to compress the KV cache — and sparse Mixture-of-Experts, which routes each token to a small subset of expert FFNs. The Engram layer adds hash-addressed n-gram memory to the residual stream without any attention overhead. Each notebook builds the technique from scratch on top of the NanoDeepSeek baseline, with honest caveats about where these gains only show up at scale.

| # | Title | Key Topics |
|---|---|---|
| DS:01 | [MLA & Mixture of Experts](/courses/llm/deepseek/01-deepseek-architecture.html) | MLA KV compression, decoupled RoPE, SwiGLU MoE, auxiliary load-balancing loss, NanoDeepSeek |
| DS:02 | [Training NanoDeepSeek](/courses/llm/deepseek/02-training-nanodeepseek.html) | FineWeb-Edu pipeline, matched-parameter GPT vs. NanoDeepSeek, MoE routing entropy |
| DS:03 | [The Engram Layer](/courses/llm/deepseek/03-engram-layer.html) | Hash-addressed n-gram memory, CompressedTokenizer, sqrt-sigmoid gating, ShortConv, NanoDeepSeek + Engram |

: {tbl-colwidths="[6,32,62]"}

See the [DeepSeek series index](/courses/llm/deepseek/index.html) to get started.

## Prerequisites

- Python 3.13+, PyTorch 2.0+, NumPy, Matplotlib
- Linear algebra, calculus, and basic probability — derivations are shown in full
- Completion of (or familiarity with) the [Deep Learning Foundations](/courses/deep-learning/index.html) series, especially [NB07 (Language Modeling)](/courses/deep-learning/07-language-modeling.html) and [NB09 (Attention & Transformers)](/courses/deep-learning/09-attention-transformers.html)
- A machine with at least one CUDA GPU is recommended for Parts II onward; Part I runs on CPU

## How to Read This Series

**If you want to understand the model architecture first:** Start with [NB01](/courses/llm/01-gpt-architecture.html) and [NB02](/courses/llm/02-tokenization.html) — these are the minimum to reason about any design decision in the rest of the series.

**If you are debugging a training run right now:** Jump to [NB04](/courses/llm/04-training-loop.html) and [NB05](/courses/llm/05-scaling-diagnostics.html). The gradient hooks and training pathologies catalog in [NB05](/courses/llm/05-scaling-diagnostics.html) is the most immediately practical entry point.

**If you are planning a pretraining run:** Work through Part II ([NB03](/courses/llm/03-data-pipelines.html)–[NB07](/courses/llm/07-distributed-training.html)) in order — data pipelines, training loop, diagnostics, mixed precision, distributed training.

**If you want to align a model:** Read Part III in order. DPO ([NB09](/courses/llm/09-preference-optimization.html)) builds on the SFT setup in ([NB08](/courses/llm/08-sft-lora.html)). GRPO ([NB10](/courses/llm/10-grpo.html)) is largely self-contained but benefits from the reward model section of [NB09](/courses/llm/09-preference-optimization.html).

**If you want to see everything working together:** Go straight to [NB15](/courses/llm/15-reasoning-model.html) — it chains every prior stage into a single end-to-end pipeline on a small, verifiable task.

**If you want the advanced architectures:** The [DeepSeek series](/courses/llm/deepseek/index.html) covers MLA, MoE, and the Engram memory layer and can be read after [NB01](/courses/llm/01-gpt-architecture.html).